In [47]:
import os
import shutil
import zipfile
import io
import numpy as np
import fitz                         
from PIL import Image
import easyocr
import openai
from dotenv import load_dotenv
import json

In [48]:
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

TEMP_DIR = "temp_uploads"
os.makedirs(TEMP_DIR, exist_ok=True)

In [49]:
ALLOWED_EXTENSIONS = {'.pdf', '.jpg', '.jpeg', '.png', '.zip'}
MAX_FILE_SIZE = 25 * 1024 * 1024  # 25 MB
ocr_reader = easyocr.Reader(['en','th'], gpu=False)

Using CPU. Note: This module is much faster with a GPU.


In [50]:
# Cell 3’: อัปเดต read_text_from_file ให้ดึง text ดิจิทัลก่อน ถ้าไม่เจอค่อย OCR
def read_text_from_file(file_path: str) -> str:
    _, ext = os.path.splitext(file_path.lower())
    text = ""
    
    if ext == '.pdf':
        doc = fitz.open(file_path)
        for page in doc:
            # พยายามดึง text ดั้งเดิมก่อน
            page_text = page.get_text("text", flags=0)
            if len(page_text.strip()) > 50:
                text += page_text + "\n"
            else:
                # ถ้า text น้อย แปลว่าคงเป็นสแกน ลง OCR
                pix = page.get_pixmap(dpi=200)
                img = Image.open(io.BytesIO(pix.tobytes(output='png')))
                arr = np.array(img)
                text += "\n".join(ocr_reader.readtext(arr, detail=0)) + "\n"
        return text
    
    if ext in {'.jpg','.jpeg','.png'}:
        arr = np.array(Image.open(file_path))
        return "\n".join(ocr_reader.readtext(arr, detail=0))
    
    if ext == '.zip':
        with zipfile.ZipFile(file_path) as z:
            for name in z.namelist():
                _, e2 = os.path.splitext(name.lower())
                if e2 in {'.jpg','.jpeg','.png'}:
                    data = z.read(name)
                    img = Image.open(io.BytesIO(data))
                    arr = np.array(img)
                    text += "\n".join(ocr_reader.readtext(arr, detail=0)) + "\n"
        return text
    
    return ""


In [51]:
# Cell 4: ฟังก์ชัน M2: Read_Text_From_File
def read_text_from_file(file_path: str) -> str:
    _, ext = os.path.splitext(file_path.lower())
    text = ""

    # กรณี ZIP
    if ext == '.zip':
        with zipfile.ZipFile(file_path) as z:
            for name in z.namelist():
                _, e2 = os.path.splitext(name.lower())
                if e2 in {'.jpg','.jpeg','.png'}:
                    data = z.read(name)
                    img = Image.open(io.BytesIO(data))
                    arr = np.array(img)
                    text += "\n".join(ocr_reader.readtext(arr, detail=0)) + "\n"
        return text

    # กรณี PDF
    if ext == '.pdf':
        doc = fitz.open(file_path)
        for page in doc:
            pix = page.get_pixmap()
            img = Image.open(io.BytesIO(pix.tobytes(output='png')))
            arr = np.array(img)
            text += "\n".join(ocr_reader.readtext(arr, detail=0)) + "\n"
        return text

    # กรณีรูปภาพ
    if ext in {'.jpg','.jpeg','.png'}:
        arr = np.array(Image.open(file_path))
        text = "\n".join(ocr_reader.readtext(arr, detail=0))
        return text

    # นามสกุลอื่น ๆ (รองรับแล้วใน M1)
    return ""


In [52]:
# Cell 5 (ปรับ prompt): บอกโมเดลว่ามีทั้งไทย–อังกฤษ
def extract_financial_fields(raw_text: str) -> dict:
    prompt = (
        "Extract the following fields from the bilingual (Thai & English) text below "
        "and return as a JSON object with keys: bill_number, supplier_name, amount, payment_date, signature.\n\n"
        f"Text:\n{raw_text}"
    )
    messages = [
        {"role":"system","content":"You are a precise data extraction assistant. The text may contain Thai and English."},
        {"role":"user","content":prompt}
    ]
    resp = openai.ChatCompletion.create(
        model="gpt-4o",
        messages=messages,
        temperature=0
    )
    content = resp.choices[0].message.content.strip()
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return {"_raw_response": content}


In [53]:
# Cell 5 (อีกครั้ง): อัปเดต extract_financial_fields ให้ใช้ new API path
import json

def extract_financial_fields(raw_text: str) -> dict:
    prompt = (
        "Extract the following fields from the bilingual (Thai & English) text below "
        "and return as a JSON object with keys: bill_number, supplier_name, amount, payment_date, signature.\n\n"
        f"Text:\n{raw_text}"
    )
    messages = [
        {"role": "system", "content": "You are a precise data extraction assistant."},
        {"role": "user",   "content": prompt}
    ]

    # เปลี่ยนจาก openai.ChatCompletion.create(...) เป็น
    resp = openai.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        temperature=0
    )
    content = resp.choices[0].message.content.strip()
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return {"_raw_response": content}


In [ ]:
# Cell X: ฟังก์ชัน M1 – อัปโหลดและตรวจสอบไฟล์
import shutil

def upload_and_validate_file(src_path: str):
    """
    ตรวจสอบนามสกุลและขนาดไฟล์ (<=25MB) 
    แล้วก็อปไฟล์มาไว้ใน TEMP_DIR ถ้าผ่าน
    """
    _, ext = os.path.splitext(src_path.lower())
    size = os.path.getsize(src_path)

    if ext not in ALLOWED_EXTENSIONS:
        return {"status":"error","message":"Unsupported file format. Accepted formats are .pdf, .jpg, .png, .zip"}
    if size > MAX_FILE_SIZE:
        return {"status":"error","message":"File is too large. File size should not exceed 25MB."}

    try:
        dest = os.path.join(TEMP_DIR, os.path.basename(src_path))
        shutil.copy(src_path, dest)
        return {"status":"success","filename":dest}
    except Exception as e:
        return {"status":"error","message":f"Upload failed: {e}"}


In [54]:
# Cell 6: ทดลองเรียกใช้ pipeline ทั้งหมดกับไฟล์ตัวอย่าง
src = r"D:\Famis-backend\FAMIS-Backend\Storage\Test1.png"
res1 = upload_and_validate_file(src)
print("Upload result:", res1)

if res1["status"] == "success":
    raw = read_text_from_file(res1["filename"])
    print("OCR text snippet:", raw[:200], "...\n")
    
    extracted = extract_financial_fields(raw)
    print("Extracted fields:", extracted)
else:
    print("Error:", res1["message"])


Upload result: {'status': 'success', 'filename': 'temp_uploads\\Test1.png'}
OCR text snippet: fm 1oog
smith machine
2310*1300*2400
85,000
85 000
fพ1022
 body staatcher
1885"700*1000
15,300
15,ว00
smd 2o09
 ad justa8le a8 benich
1380 7วด 4..
17,300
เร.วาา
nx5 840
3 naax ad justable bench iflat
 ...



AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-or-v1*************************************************************1d08. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}